# Launch & finish DPS runs of `run_boreal_biomass_map` (v2)

Submits DPS jobs that map boreal **vegetation height** (`Ht`) or **AGB** for a list of years, finds which tiles are still **missing** or **failed**, and resubmits only those. After jobs finish, it rebuilds the tile index (tindex) that defines what "complete" means.

**Run top to bottom. The only cell you edit is `1. Run configuration`.** Nothing is submitted unless `DRY_RUN = False`.

| Step | What it does | Touches DPS? |
|---|---|---|
| 2. Status | Reads each year's tindex: tiles **have** vs **missing** | no |
| 3. Fails | Latest job per tile from saved submission logs → which tiles **failed** (and are still not in the tindex) | reads job status |
| 4. Plan | Per year: tiles to submit (`DO_MISSING` / `DO_FAILS`), minus tiles that lack inputs. Saved as CSV. | no |
| 5. Submit | Submits the plan (only if `DRY_RUN = False`) | **yes** |
| 6. Assess | Status counts for the submission just made | reads job status |
| 7. Rebuild tindex | After jobs finish: backs up, rebuilds from **all** run dates, verifies the count did not shrink | no |

**Typical cycle to finish a product**: `DO_MISSING=True` → plan → submit → wait → 6 → 7 → back to 2. Then `DO_FAILS=True` for stragglers, or keep `DO_MISSING` (missing ⊇ failed once the tindex is rebuilt).

**`DO_MISSING` vs `DO_FAILS`**
* **MISSING** = in `TILE_SET` but not in the tindex. It covers never-submitted tiles *and* failed tiles, but only as current as the last tindex rebuild (step 7).
* **FAILS** = the tile's most recent job for this identifier has status `Failed` (or never submitted successfully) *and* the tile is not in the tindex. Use it right after a run, before rebuilding the tindex, or to retry only known failures.
* Both: union. Neither: every tile in `TILE_SET` (a full re-run, which needs `CONFIRM_FULL_RUN = True`).

Ali Zarringhalam, PhD | Paul Montesano, PhD — reorganized Oct 2026 from `DPS_run_boreal.ipynb`

In [2]:
import os, sys, glob, json, shutil, datetime, subprocess
from functools import lru_cache
import pandas as pd
import geopandas as gpd

from maap.maap import MAAP
maap = MAAP(maap_host='api.maap-project.org')

# One clone for imports AND for the build_tindex script. (The old notebook appended
# my-private-bucket/code but actually imported /projects/code; this makes the choice explicit.)
ICESAT2_BOREAL_REPO_PATH = '/projects/my-private-bucket/code/icesat2_boreal'
sys.path.insert(0, ICESAT2_BOREAL_REPO_PATH + '/lib')
import ExtractUtils
import mosaiclib
from mosaiclib import (HLS_TINDEX_FN_DICT, TOPO_TINDEX_FN_DICT, LC_TINDEX_FN_DICT, SAR_TINDEX_FN_DICT,
                       ATL08_FILT_EXTRACT_TINDEX_FN_DICT, BOREAL_TRAIN_DATA_FN_DICT, boreal_tile_index_path)
print('ExtractUtils from', ExtractUtils.__file__)

NASA MAAP
ExtractUtils from /projects/my-private-bucket/code/icesat2_boreal/lib/ExtractUtils.py


## 1. Run configuration — the only cell to edit
Fill-in runs **must** use the same `VERSION`, `SUB_IDENTIFIER`, `ALGO_PARAMS` and inputs as the original run, so the new tiles are consistent with the rest of the map.

In [41]:
# --- What product -----------------------------------------------------------
ALGO_ID      = 'run_boreal_biomass_map'
VERSION      = 'v3.1.0'
USER         = 'montesano'
WORKER_TYPE  = 'maap-dps-worker-32vcpu-64gb'
PREDICT_VAR  = 'Ht'                      # 'Ht' or 'AGB' (case sensitive: becomes part of the DPS identifier)
DO_MULTIYEAR = True                      # multi-year ATL08 training (2019-2024 merged)
SUB_IDENTIFIER = 'full_run_niter30_multiyear_atl08v6' if DO_MULTIYEAR else 'full_run_niter30_singleyear_atl08v6'
YEAR_LIST    = [str(y) for y in range(2016, 2018 + 1)] + [2025]

# --- Which tiles -------------------------------------------------------------
DO_MISSING = True        # tiles in TILE_SET not yet in the tindex
DO_FAILS   = False       # tiles whose latest job failed and are still not in the tindex
CONFIRM_FULL_RUN = False # must be True to submit every tile in TILE_SET (both flags False)
TILE_SET = None          # None = all boreal tiles; or a list of tile_num, e.g. mosaiclib.set_agbd_change_demo_tiles
REFRESH_FAILS = True     # re-query job status (slow for many jobs); False = reuse the cached fails JSON

# --- Safety -----------------------------------------------------------------
DRY_RUN = False           # True: build and save the plan only. False: submit in step 5.
MAX_JOBS = 6000          # refuse to submit more than this in one go

# --- Algorithm parameters (copied unchanged from the original run) -----------
INPUT_IMAGERY = ('hls', 'topo', 'lc')    # add 'sar' to use Sentinel-1 where available
ATL08_YEARS = '2019 2020 2021 2022 2023 2024'
ALGO_PARAMS = {
    'DO_SLOPE_VALID_MASK': True, 'n_iters': 30, 'ntree': 50, 'zero_short_veg_height': True, 'cores': 30,
    'slope_thresh': 15, 'in_tile_field': 'tile_num', 'minDOY': 130, 'maxDOY': 250, 'max_sol_el': 5,
    'expand_training': True, 'local_train_perc': 90, 'min_samples': 5000, 'max_samples': 10000,
    'pred_vars': ('Red Green elevation slope tsri tpi NIR SWIR SWIR2 NDVI SAVI MSAVI NDMI EVI NBR NBR2 TCB TCG TCW '
                  'vv_median_frozen vh_median_frozen vv_median_summer vh_median_summer vv_median_shoulder vh_median_shoulder'),
}

# --- Paths --------------------------------------------------------------------
DPS_SUBMISSION_RESULTS_DIR = '/projects/my-public-bucket/dps_submission_results'
TINDEX_ROOT   = '/projects/my-public-bucket/DPS_tile_lists/BOREAL_MAP'      # build_tindex output root
DPS_OUTPUT_ROOT = f'/projects/my-private-bucket/dps_output/{ALGO_ID}/{VERSION}'
PLAN_DIR      = f'{DPS_SUBMISSION_RESULTS_DIR}/plans'
for d in (DPS_SUBMISSION_RESULTS_DIR, PLAN_DIR):
    os.makedirs(d, exist_ok=True)

In [43]:
# Derived names (do not edit)
TINDEX_TYPE = PREDICT_VAR.upper()                                   # 'HT' or 'AGB' (build_tindex -t)
IDENTIFIERS = {y: f'{PREDICT_VAR}_H30_{y}/{SUB_IDENTIFIER}' for y in YEAR_LIST}
MODE = '+'.join([m for m, on in (('MISSING', DO_MISSING), ('FAILS', DO_FAILS)) if on]) or 'ALL'
print(f'{ALGO_ID} {VERSION} | {PREDICT_VAR} | years {YEAR_LIST[0]}–{YEAR_LIST[-1]} | mode {MODE} | DRY_RUN={DRY_RUN}')
for y, i in IDENTIFIERS.items():
    print('  ', i)

run_boreal_biomass_map v3.1.0 | Ht | years 2016–2025 | mode MISSING | DRY_RUN=False
   Ht_H30_2016/full_run_niter30_multiyear_atl08v6
   Ht_H30_2017/full_run_niter30_multiyear_atl08v6
   Ht_H30_2018/full_run_niter30_multiyear_atl08v6
   Ht_H30_2025/full_run_niter30_multiyear_atl08v6


### Helper functions (no edits needed)
Parameter builder: unchanged logic from the original notebook (`get_in_param_dict`), now reading `ALGO_PARAMS`.

In [44]:
def tindex_paths(identifier):
    d = f'{TINDEX_ROOT}/{VERSION}/{identifier}'
    return {'dir': d, 'csv': f'{d}/{TINDEX_TYPE}_tindex_master.csv', 'gpkg': f'{d}/{TINDEX_TYPE}_tindex_master.gpkg'}

def get_tindex_fn_for(product, multiyear_atl08=False, year=None):
    return {
        # multi-year: 2020 is a placeholder; the alg merges ATL08_YEARS (TODO: make atl08 tindex optional in the yml)
        'atl08': ATL08_FILT_EXTRACT_TINDEX_FN_DICT['2020'] if multiyear_atl08 else ATL08_FILT_EXTRACT_TINDEX_FN_DICT[year],
        'hls':  HLS_TINDEX_FN_DICT[year],
        'topo': TOPO_TINDEX_FN_DICT['c2020updated_v2'],
        'lc':   LC_TINDEX_FN_DICT['c2020updated'],
        'sar':  SAR_TINDEX_FN_DICT.get(year, None),
    }[product]

@lru_cache(maxsize=None)
def get_tindex_df_for(products, multiyear_atl08=False, year=None):
    return {p: pd.read_csv(get_tindex_fn_for(p, multiyear_atl08, year)).set_index('tile_num') for p in products}

def get_fn_url_pair_for_tile(tindex, product, tile_num):
    url = tindex[product].loc[tile_num, 's3_path']
    if isinstance(url, pd.Series):          # duplicate tile rows in an input tindex: take the last
        url = url.iloc[-1]
    return {f'in_{product}_fn': f'input/{os.path.basename(url)}', f'in_{product}_fn_url': url}

def get_const_url_fn_pair(year, multiyear_atl08=False):
    const_urls = {
        'in_tile_fn_url': 's3://maap-ops-workspace/shared/montesano/databank/boreal_tiles_v004_model_ready.gpkg',
        'boreal_vect': 's3://maap-ops-workspace/shared/nathanmthomas/analyze_agb/input_zones/wwf_circumboreal_Dissolve.geojson',
        'ecoregions_fn_url': 's3://maap-ops-workspace/aliz237/data/vectors/wwf_terr_ecos.gpkg',
        'in_atl08_fn_url': get_tindex_fn_for('atl08', multiyear_atl08, year),
        'in_atl08_sample_url': BOREAL_TRAIN_DATA_FN_DICT['merged'] if multiyear_atl08 else BOREAL_TRAIN_DATA_FN_DICT[year],
        'bio_models_tar_fn_url': 's3://maap-ops-workspace/aliz237/biomass_models/bio_models_noground.tar',
    }
    fns = {}
    for k, v in const_urls.items():
        fn_key = k.replace('_url', '')
        fn_key = fn_key if fn_key.endswith('_fn') else f'{fn_key}_fn'
        fns[fn_key] = f'input/{os.path.basename(v)}'
    const_urls.update(fns)
    return const_urls

def get_in_param_dict(tile_num, year, target, multiyear_atl08=False):
    tindex = get_tindex_df_for(tuple(INPUT_IMAGERY), multiyear_atl08, year)
    p = {}
    for product in INPUT_IMAGERY:
        p.update(get_fn_url_pair_for_tile(tindex, product, tile_num))
    p.update(get_const_url_fn_pair(year, multiyear_atl08))
    p.update(ALGO_PARAMS)
    p.update({'in_tile_num': tile_num, 'predict_var': target, 'year': year,
              'atl08_year_list': ATL08_YEARS if multiyear_atl08 else str(year)})
    if 'in_sar_fn_url' in p:
        p['in_sar_fn'] = p.pop('in_sar_fn_url')
    return p

In [45]:
def read_tindex_tiles(identifier):
    """Tiles present in the tindex for this identifier (empty set if no tindex yet)."""
    tp = tindex_paths(identifier)
    for fn in (tp['csv'], tp['gpkg']):
        if os.path.exists(fn):
            df = pd.read_csv(fn) if fn.endswith('.csv') else gpd.read_file(fn)
            return set(df.tile_num.astype(int))
    return set()

def submission_logs(identifier):
    """All saved submission CSVs for this identifier, oldest first (by the timestamp in the file name)."""
    s = identifier.replace('/', '_')
    fns = glob.glob(f'{DPS_SUBMISSION_RESULTS_DIR}/DPS_{ALGO_ID}_{s}_submission_results_*.csv')
    return sorted(fns, key=lambda f: int(os.path.splitext(f)[0].split('_')[-1]))

def latest_jobs(identifier):
    """One row per tile: the most recent submission for this identifier."""
    parts = []
    for order, fn in enumerate(submission_logs(identifier)):
        d = pd.read_csv(fn)
        d['log_file'], d['log_order'] = os.path.basename(fn), order
        parts.append(d)
    if not parts:
        return pd.DataFrame(columns=['tile_num', 'job_id', 'submit_status', 'log_file'])
    d = pd.concat(parts, ignore_index=True)
    d['tile_num'] = d.tile_num.astype(int)
    return d.sort_values(['log_order', 'dps_num']).drop_duplicates('tile_num', keep='last').reset_index(drop=True)

def job_status_table(jobs):
    """Adds a 'status' column via MAAP. Tiles that never got a job id are marked 'SubmitFailed'."""
    ok = jobs.job_id.notna() & (jobs.job_id.astype(str) != '')
    out = jobs.copy()
    out['status'] = 'SubmitFailed'
    if ok.any():
        st = ExtractUtils.BUILD_TABLE_JOBSTATUS(jobs[ok][['tile_num', 'job_id']])
        out.loc[ok, 'status'] = out.loc[ok, 'job_id'].map(dict(zip(st.job_id, st.status))).fillna('Unknown')
    return out

In [46]:
def build_plan(year, identifier, tile_set, have, failed):
    """Tiles to submit for one identifier, with the reason each tile is (or is not) included."""
    missing = sorted(set(tile_set) - have)
    want = set()
    if DO_MISSING:
        want |= set(missing)
    if DO_FAILS:
        want |= set(failed)
    if not (DO_MISSING or DO_FAILS):
        want = set(tile_set)
    rows = []
    for t in sorted(want):
        try:
            get_in_param_dict(t, year, PREDICT_VAR, multiyear_atl08=DO_MULTIYEAR)
            runnable, why = True, ''
        except KeyError as e:
            runnable, why = False, f'no input for tile in {e}'
        rows.append({'identifier': identifier, 'year': year, 'tile_num': t,
                     'is_missing': t in missing, 'is_failed': t in failed, 'in_tindex': t in have,
                     'runnable': runnable, 'reason_not_runnable': why})
    return pd.DataFrame(rows, columns=['identifier', 'year', 'tile_num', 'is_missing', 'is_failed', 'in_tindex', 'runnable', 'reason_not_runnable'])

def submit_identifier(identifier, year, tiles):
    rows = []
    for i, t in enumerate(tiles, 1):
        try:
            r = maap.submitJob(identifier=identifier, algo_id=ALGO_ID, version=VERSION, username=USER, queue=WORKER_TYPE,
                               **get_in_param_dict(t, year, PREDICT_VAR, multiyear_atl08=DO_MULTIYEAR))
            job_id, status = r.id, r.status
        except Exception as e:
            job_id, status = None, f'error: {e}'
        rows.append({'dps_num': i, 'tile_num': t, 'submit_time': datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S'),
                     'algo_id': ALGO_ID, 'user': USER, 'worker_type': WORKER_TYPE, 'job_id': job_id,
                     'submit_status': status, 'IDENTIFIER': identifier})
        if i in (1, len(tiles)) or i % 100 == 0:
            print(f'  {i}/{len(tiles)} tile {t}: {status}')
    df = pd.DataFrame(rows)
    now = pd.Timestamp.now().strftime('%Y%m%d%H%M')
    fn = f"{DPS_SUBMISSION_RESULTS_DIR}/DPS_{ALGO_ID}_{identifier.replace('/', '_')}_submission_results_{len(df)}_{now}.csv"
    df.to_csv(fn, index=False)
    print(f'  saved {fn}')
    return df

In [47]:
def dps_run_dates(identifier):
    """Years, months and day range of every DPS run-date folder for this identifier:
    dps_output/<alg>/<version>/<identifier>/YYYY/MM/DD."""
    root = f'{DPS_OUTPUT_ROOT}/{identifier}'
    years, months, days = set(), set(), set()
    if os.path.isdir(root):
        for y in os.listdir(root):
            if not (y.isdigit() and os.path.isdir(f'{root}/{y}')):
                continue
            for m in os.listdir(f'{root}/{y}'):
                if not (m.isdigit() and os.path.isdir(f'{root}/{y}/{m}')):
                    continue
                for d in os.listdir(f'{root}/{y}/{m}'):
                    if d.isdigit():
                        years.add(y); months.add(m); days.add(int(d))
    return sorted(years), sorted(months), (min(days), max(days)) if days else (1, 31)

def rebuild_tindex(identifier, ENV='pangeo', DEBUG=False,
                   SCRIPT=f'{ICESAT2_BOREAL_REPO_PATH}/lib/build_tindex_master_refactored.py'):
    """Back up, rebuild with build_tindex_master_refactored.py from ALL run dates, verify the tile count.

    The script OVERWRITES {TYPE}_tindex_master.csv/.gpkg with only the outputs found under the run dates
    it is given (HT/AGB search: <user>/dps_output/<alg>/<VERSION>/<identifier>/{year}/{month}/{day}/**/*_NNNNNNN.tif),
    keeping the newest file per tile. So every run date must be passed, not just the latest fill-in run.
    (--tindex_append is not used: it concatenates without de-duplicating.)
    Years x months x days are searched as a cartesian product; the day range is trimmed to what exists."""
    tp = tindex_paths(identifier)
    years, months, (d_min, d_max) = dps_run_dates(identifier)
    if not years:
        print(f'  no DPS output folders under {DPS_OUTPUT_ROOT}/{identifier}; skipping')
        return None
    n_before = len(read_tindex_tiles(identifier))
    stamp = pd.Timestamp.now().strftime('%Y%m%d%H%M%S')
    bdir = f"{tp['dir']}/backup_{stamp}"
    for fn in (tp['csv'], tp['gpkg']):
        if os.path.exists(fn):
            os.makedirs(bdir, exist_ok=True)
            shutil.copy2(fn, bdir)
    os.makedirs(tp['dir'], exist_ok=True)
    cmd = [f'/opt/conda/envs/{ENV}/bin/python', SCRIPT,
           '-t', TINDEX_TYPE, '-alg_name', ALGO_ID, '--user', USER,
           '--dps_identifier', f'{VERSION}/{identifier}',
           '-y_list', *years, '-m_list', *months, '-d_min', str(d_min), '-d_max', str(d_max),
           '--outdir', tp['dir'], '-boreal_tile_index_path', boreal_tile_index_path,
           '--RETURN_DUPS', '--WRITE_TINDEX_MATCHES_GDF']
    if DEBUG:
        cmd.append('--DEBUG')
    print(f'  run dates: years {years} months {months} days {d_min}-{d_max}')
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:   # the script exits 1 before writing when nothing is found, so the tindex is untouched
        print(res.stdout[-3000:], res.stderr[-3000:])
        raise RuntimeError(f'{os.path.basename(SCRIPT)} failed for {identifier}; backup in {bdir}')
    if DEBUG:
        print(res.stdout[-3000:])
    n_after = len(read_tindex_tiles(identifier))
    flag = 'OK' if n_after >= n_before else '!! SHRANK — restore with restore_tindex(identifier, backup_dir)'
    print(f'  tiles before {n_before} -> after {n_after}  {flag}   (backup: {bdir if n_before else "none"})')
    return {'identifier': identifier, 'n_before': n_before, 'n_after': n_after, 'backup_dir': bdir}

def restore_tindex(identifier, backup_dir):
    for fn in glob.glob(f'{backup_dir}/*'):
        shutil.copy2(fn, tindex_paths(identifier)['dir'])
    print('restored', identifier, 'from', backup_dir)

## 2. Status: which tiles each year has and is missing

In [48]:
boreal_tiles = gpd.read_file(boreal_tile_index_path)
TILES = sorted(set(int(t) for t in (TILE_SET if TILE_SET is not None else boreal_tiles.tile_num)))
HAVE = {y: read_tindex_tiles(i) for y, i in IDENTIFIERS.items()}
status = pd.DataFrame([{'year': y, 'identifier': IDENTIFIERS[y], 'n_target': len(TILES),
                        'n_have': len(set(TILES) & HAVE[y]), 'n_missing': len(set(TILES) - HAVE[y])} for y in YEAR_LIST])
status

,year,identifier,n_target,n_have,n_missing
0,2016,Ht_H30_2016/full_run_niter30_multiyear_atl08v6,4956,647,4309
1,2017,Ht_H30_2017/full_run_niter30_multiyear_atl08v6,4956,646,4310
2,2018,Ht_H30_2018/full_run_niter30_multiyear_atl08v6,4956,647,4309
3,2025,Ht_H30_2025/full_run_niter30_multiyear_atl08v6,4956,646,4310


In [39]:
# Optional: map of missing tiles for one year
SHOW_MAP_YEAR = YEAR_LIST[-1]
boreal_tiles[boreal_tiles.tile_num.isin(set(TILES) - HAVE[SHOW_MAP_YEAR])].explore(color='red', tooltip='tile_num')

## 3. Fails: latest job per tile from the saved submission logs
Only tiles whose **most recent** job for the identifier failed (or never got a job id), and that are still not in the tindex. Cached to JSON so `DO_FAILS` works when you come back without re-querying (`REFRESH_FAILS = False`).

In [25]:
FAILS_JSON = f"{PLAN_DIR}/fails_{ALGO_ID}_{VERSION}_{PREDICT_VAR}_{SUB_IDENTIFIER}.json"
if DO_FAILS and not REFRESH_FAILS and os.path.exists(FAILS_JSON):
    FAILED = {k: set(v) for k, v in json.load(open(FAILS_JSON)).items()}
    print('loaded cached fails from', FAILS_JSON)
elif DO_FAILS:
    FAILED, job_tables = {}, {}
    for y, ident in IDENTIFIERS.items():
        jobs = latest_jobs(ident)
        if jobs.empty:
            FAILED[y] = set(); print(f'{ident}: no submission logs'); continue
        jt = job_status_table(jobs)
        job_tables[y] = jt
        bad = jt.status.isin(['Failed', 'SubmitFailed', 'Offline', 'Deleted'])
        FAILED[y] = set(jt.tile_num[bad].astype(int)) - HAVE[y]
        print(f'{ident}: latest jobs {len(jt)} | ' + ', '.join(f'{k} {v}' for k, v in jt.status.value_counts().items())
              + f' | failed & not in tindex: {len(FAILED[y])}')
    json.dump({y: sorted(v) for y, v in FAILED.items()}, open(FAILS_JSON, 'w'))
else:
    FAILED = {y: set() for y in YEAR_LIST}
    print('DO_FAILS = False: fails not computed')

DO_FAILS = False: fails not computed


## 4. Plan: what will be submitted
Tiles without an input (e.g. no HLS composite that year) are listed as not runnable and are **not** submitted, so they don't fail repeatedly. Review this table before setting `DRY_RUN = False`.

In [26]:
if not (DO_MISSING or DO_FAILS) and not CONFIRM_FULL_RUN:
    raise ValueError('Both DO_MISSING and DO_FAILS are False (full re-run of TILE_SET). Set CONFIRM_FULL_RUN = True if intended.')

plan = pd.concat([build_plan(y, IDENTIFIERS[y], TILES, HAVE[y], FAILED.get(y, set())) for y in YEAR_LIST], ignore_index=True)
plan = plan.astype({'tile_num': int, 'is_missing': bool, 'is_failed': bool, 'in_tindex': bool, 'runnable': bool})
plan_summary = (plan.groupby(['year', 'identifier'])
                    .agg(n_selected=('tile_num', 'size'), n_missing=('is_missing', 'sum'), n_failed=('is_failed', 'sum'),
                         n_not_runnable=('runnable', lambda s: int((~s).sum())), n_to_submit=('runnable', 'sum'))
                    .reset_index())
PLAN_FN = f"{PLAN_DIR}/plan_{PREDICT_VAR}_{SUB_IDENTIFIER}_{MODE}_{pd.Timestamp.now():%Y%m%d%H%M}.csv"
plan.to_csv(PLAN_FN, index=False)
print('plan saved:', PLAN_FN, '| total to submit:', int(plan.runnable.sum()))
plan_summary

plan saved: /projects/my-public-bucket/dps_submission_results/plans/plan_Ht_full_run_niter30_multiyear_atl08v6_MISSING_202610100941.csv | total to submit: 291


,year,identifier,n_selected,n_missing,n_failed,n_not_runnable,n_to_submit
0,2019,Ht_H30_2019/full_run_niter30_multiyear_atl08v6,51,51,0,1,50
1,2020,Ht_H30_2020/full_run_niter30_multiyear_atl08v6,49,49,0,1,48
2,2021,Ht_H30_2021/full_run_niter30_multiyear_atl08v6,49,49,0,1,48
3,2022,Ht_H30_2022/full_run_niter30_multiyear_atl08v6,50,50,0,1,49
4,2023,Ht_H30_2023/full_run_niter30_multiyear_atl08v6,49,49,0,1,48
5,2024,Ht_H30_2024/full_run_niter30_multiyear_atl08v6,49,49,0,1,48


In [27]:
# Not-runnable tiles and why (fix inputs first, or accept as permanent gaps)
plan[~plan.runnable].groupby(['year', 'reason_not_runnable']).tile_num.apply(lambda s: sorted(s)).reset_index()

,year,reason_not_runnable,tile_num
0,2019,no input for tile in 45970,[45970]
1,2020,no input for tile in 45970,[45970]
2,2021,no input for tile in 45970,[45970]
3,2022,no input for tile in 45970,[45970]
4,2023,no input for tile in 45970,[45970]
5,2024,no input for tile in 45970,[45970]


## 5. Submit (only if `DRY_RUN = False`)

In [28]:
SUBMITTED = {}
n_total = int(plan.runnable.sum())
if DRY_RUN:
    print(f'DRY_RUN: would submit {n_total} jobs. Set DRY_RUN = False in section 1 and re-run from the top.')
elif n_total > MAX_JOBS:
    raise ValueError(f'{n_total} jobs > MAX_JOBS={MAX_JOBS}. Raise MAX_JOBS if intended.')
else:
    for y, ident in IDENTIFIERS.items():
        tiles = plan[(plan.year == y) & plan.runnable].tile_num.tolist()
        print(f'\n{ident}: {len(tiles)} jobs')
        if tiles:
            SUBMITTED[y] = submit_identifier(ident, y, tiles)


Ht_H30_2019/full_run_niter30_multiyear_atl08v6: 50 jobs
  1/50 tile 1093: success
  50/50 tile 23827: success
  saved /projects/my-public-bucket/dps_submission_results/DPS_run_boreal_biomass_map_Ht_H30_2019_full_run_niter30_multiyear_atl08v6_submission_results_50_202610100941.csv

Ht_H30_2020/full_run_niter30_multiyear_atl08v6: 48 jobs
  1/48 tile 1735: success
  48/48 tile 23827: success
  saved /projects/my-public-bucket/dps_submission_results/DPS_run_boreal_biomass_map_Ht_H30_2020_full_run_niter30_multiyear_atl08v6_submission_results_48_202610100941.csv

Ht_H30_2021/full_run_niter30_multiyear_atl08v6: 48 jobs
  1/48 tile 1735: success
  48/48 tile 23827: success
  saved /projects/my-public-bucket/dps_submission_results/DPS_run_boreal_biomass_map_Ht_H30_2021_full_run_niter30_multiyear_atl08v6_submission_results_48_202610100941.csv

Ht_H30_2022/full_run_niter30_multiyear_atl08v6: 49 jobs
  1/49 tile 1735: success
  49/49 tile 23827: success
  saved /projects/my-public-bucket/dps_subm

## 6. Assess the submission just made
Re-run this cell until nothing is `Accepted`/`Running`. To assess an older run in a new session, set `DO_FAILS = True` and run section 3, which reads the saved logs.

In [49]:
if SUBMITTED:
    assess = pd.concat([job_status_table(df) for df in SUBMITTED.values()], ignore_index=True)
    display(assess.groupby(['IDENTIFIER', 'status']).size().unstack(fill_value=0))
else:
    print('Nothing submitted in this session.')

multiprocessing...
Count total jobs:	50
Count pending jobs:	0
Count running jobs:	0
Count succeeded jobs:	46
Count failed jobs:	4
% of failed jobs:	8.0

multiprocessing...
Count total jobs:	48
Count pending jobs:	0
Count running jobs:	0
Count succeeded jobs:	42
Count failed jobs:	6
% of failed jobs:	12.5

multiprocessing...
Count total jobs:	47
Count pending jobs:	0
Count running jobs:	0
Count succeeded jobs:	43
Count failed jobs:	4
% of failed jobs:	8.51

multiprocessing...
Count total jobs:	49
Count pending jobs:	0
Count running jobs:	0
Count succeeded jobs:	45
Count failed jobs:	4
% of failed jobs:	8.16

multiprocessing...
Count total jobs:	48
Count pending jobs:	0
Count running jobs:	0
Count succeeded jobs:	44
Count failed jobs:	4
% of failed jobs:	8.33

multiprocessing...
Count total jobs:	48
Count pending jobs:	0
Count running jobs:	0
Count succeeded jobs:	43
Count failed jobs:	5
% of failed jobs:	10.42



status,Failed,SubmitFailed,Succeeded
IDENTIFIER,,,
Ht_H30_2019/full_run_niter30_multiyear_atl08v6,4,0,46
Ht_H30_2020/full_run_niter30_multiyear_atl08v6,6,0,42
Ht_H30_2021/full_run_niter30_multiyear_atl08v6,4,1,43
Ht_H30_2022/full_run_niter30_multiyear_atl08v6,4,0,45
Ht_H30_2023/full_run_niter30_multiyear_atl08v6,4,0,44
Ht_H30_2024/full_run_niter30_multiyear_atl08v6,5,0,43


## 7. Rebuild the tindex (after jobs finish)
Uses **all** run-date folders found for each identifier, backs up the existing tindex first, and reports before/after tile counts. Then go back to section 2.

In [40]:
DO_REBUILD_TINDEX = True
if DO_REBUILD_TINDEX:
    rebuilt = [rebuild_tindex(i) for i in IDENTIFIERS.values()]
    display(pd.DataFrame([r for r in rebuilt if r]))

  run dates: years ['2025', '2026'] months ['09', '10'] days 10-24
  tiles before 4951 -> after 4951  OK   (backup: /projects/my-public-bucket/DPS_tile_lists/BOREAL_MAP/v3.1.0/Ht_H30_2019/full_run_niter30_multiyear_atl08v6/backup_20261010102505)
  run dates: years ['2025', '2026'] months ['09', '10'] days 10-24
  tiles before 4946 -> after 4947  OK   (backup: /projects/my-public-bucket/DPS_tile_lists/BOREAL_MAP/v3.1.0/Ht_H30_2020/full_run_niter30_multiyear_atl08v6/backup_20261010102609)
  run dates: years ['2025', '2026'] months ['09', '10'] days 10-23
  tiles before 4950 -> after 4950  OK   (backup: /projects/my-public-bucket/DPS_tile_lists/BOREAL_MAP/v3.1.0/Ht_H30_2021/full_run_niter30_multiyear_atl08v6/backup_20261010102702)
  run dates: years ['2025', '2026'] months ['09', '10'] days 10-23
  tiles before 4950 -> after 4950  OK   (backup: /projects/my-public-bucket/DPS_tile_lists/BOREAL_MAP/v3.1.0/Ht_H30_2022/full_run_niter30_multiyear_atl08v6/backup_20261010102757)
  run dates: yea

,identifier,n_before,n_after,backup_dir
0,Ht_H30_2019/full_run_niter30_multiyear_atl08v6,4951,4951,/projects/my-public-bucket/DPS_tile_lists/BORE...
1,Ht_H30_2020/full_run_niter30_multiyear_atl08v6,4946,4947,/projects/my-public-bucket/DPS_tile_lists/BORE...
2,Ht_H30_2021/full_run_niter30_multiyear_atl08v6,4950,4950,/projects/my-public-bucket/DPS_tile_lists/BORE...
3,Ht_H30_2022/full_run_niter30_multiyear_atl08v6,4950,4950,/projects/my-public-bucket/DPS_tile_lists/BORE...
4,Ht_H30_2023/full_run_niter30_multiyear_atl08v6,4951,4951,/projects/my-public-bucket/DPS_tile_lists/BORE...
5,Ht_H30_2024/full_run_niter30_multiyear_atl08v6,4950,4950,/projects/my-public-bucket/DPS_tile_lists/BORE...


---
## Appendix: reference calls (not part of the run)

In [34]:
SHOW_REFERENCE = True
if SHOW_REFERENCE:
    print([a for a in maap.listAlgorithms().json()['algorithms'] if ALGO_ID in a['type']])
    print(maap.getQueues().json()['queues'])
    # Register the algorithm (only after changing the yml / code):
    # maap.register_algorithm_from_yaml_file(f'{ICESAT2_BOREAL_REPO_PATH}/dps/registered/run_boreal_biomass.yml').text
    # Inspect the full parameter dict for one tile:
    print(json.dumps(get_in_param_dict(TILES[0], YEAR_LIST[0], PREDICT_VAR, multiyear_atl08=DO_MULTIYEAR), indent=1, default=str))

[{'type': 'run_boreal_biomass_map', 'version': 'dev_v1.5'}, {'type': 'run_boreal_biomass_map', 'version': 'v3.1.0'}, {'type': 'run_boreal_biomass_map', 'version': 'CA-only-bio-models'}, {'type': 'run_boreal_biomass_map', 'version': 'H30_v1.0'}, {'type': 'run_boreal_biomass_map', 'version': 'fire'}, {'type': 'run_boreal_biomass_map', 'version': 'test-ranger'}, {'type': 'run_boreal_biomass_map', 'version': 'dev_v1.14'}, {'type': 'run_boreal_biomass_map', 'version': 'dev_v1.9'}, {'type': 'run_boreal_biomass_map', 'version': 'dev_v1.16'}, {'type': 'run_boreal_biomass_map', 'version': 'predict-rh50'}, {'type': 'run_boreal_biomass_map', 'version': 'boreal_agb_2023_v2'}, {'type': 'run_boreal_biomass_map', 'version': 'dev_v1.4'}, {'type': 'run_boreal_biomass_map', 'version': 'dev_v1.15'}, {'type': 'run_boreal_biomass_map', 'version': 'boreal_summary'}, {'type': 'run_boreal_biomass_map', 'version': 'dev_v2.1'}, {'type': 'run_boreal_biomass_map', 'version': 'boreal_agb_2023_v3'}, {'type': 'run_b